In [1]:
import gpt as g
import sys, os
import numpy as np

from scipy.linalg import expm
import time
import matplotlib.pyplot as plt
from gpt.qcd.gauge.smear import local_stout  
from gpt.ad import reverse as rad
from gpt.qcd.gauge.smear.differentiable import dft_diffeomorphism
import time

SharedMemoryNone: SharedMemoryAllocate 1073741824 GPU implementation 
0SharedMemoryNone:  SharedMemoryNone.cc acceleratorAllocDevice 1073741824bytes at 0x300000000 for comms buffers 

__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|_ |  |  |  |  |  |  |  |  |  |  |  | _|__
__|_                                    _|__
__|_   GGGG    RRRR    III    DDDD      _|__
__|_  G        R   R    I     D   D     _|__
__|_  G        R   R    I     D    D    _|__
__|_  G  GG    RRRR     I     D    D    _|__
__|_  G   G    R  R     I     D   D     _|__
__|_   GGGG    R   R   III    DDDD      _|__
__|_                                    _|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
__|__|__|__|__|__|__|__|__|__|__|__|__|__|__
  |  |  |  |  |  |  |  |  |  |  |  |  |  |  


Copyright (C) 2015 Peter Boyle, Azusa Yamaguchi, Guido Cossu, Antonin Portelli and other authors

This program is free software; you can redistribute it and/or modify
it under the term

In [2]:
def trace_U(U):                                                      
    return sum(v for v in sum(u[:].real for u in g.eval(g.trace(U)))) / (4 * size**4 ) / 3.

In [3]:
num_steps = 1
def ftg(U, eps):
    global num_steps, fm

    ##### dmuAmu ##############
    #B = U[0] -  g.adj(g.cshift(U[0], 0, -1))
    B = U[0] -  g.cshift(U[0], 0, -1)
    for mu in [1,2,3,]:
        #B += U[mu] -  g.adj(g.cshift(U[mu], mu, -1))
        B += U[mu] -  g.cshift(U[mu], mu, -1)
        

   #### masks for all even/odd sites ########
    """
    grid_cb = grid.checkerboarded(g.redblack)
    one_cb = g.complex(grid_cb)
    one_cb[:] = 1

    masks = {}
    for p in [g.even, g.odd]:
        m = g.complex(grid)
        m[:] = 0
        one_cb.checkerboard(p)
        g.set_checkerboard(m, one_cb)
        masks[p] = m
    
    if num_steps // 2 == 0:
        mask, imask = masks[g.odd], masks[g.odd.inv()] 
    else:
        mask, imask = masks[g.even], masks[g.even.inv()]
    
    num_steps += 1
    fm = g(mask + 1e-15 * imask)
    #fm = mask + 1e-15 * imask
    
    ###### apply masks ########
    #B *= fm # causes issues with action log det routine
    #B = g(B*fm)
    """
    
    # apply gtf 
    U_prime = []
    for mu in [0,1,2,3]:
        U_mu_prime = g(
                g.matrix.exp(  - eps * g.qcd.gauge.project.traceless_anti_hermitian(B) )  
                * U[mu] * g.matrix.exp(  + eps * g.qcd.gauge.project.traceless_anti_hermitian( g.cshift(B, mu, +1) ) ) 
        )
        U_prime.append(U_mu_prime)
    
    return U_prime


In [23]:
size = 4
grid = g.grid([size, size, size, size], g.double)
rng = g.random("t")

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)


GPT :    5584.159466 s : Initializing gpt.random(t,vectorized_ranlux24_389_64) took 0.00018096 s


[lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double)]

In [24]:
def ft0(U):
    return ftg(U, eps=1e-2)
    #return gtf(U)

ft0(U)
    
dft = dft_diffeomorphism(U, ft0)
U1 = dft(U)

V = g.mcolor(grid) # V is a SU(3) matrix on every lattice site
rng.normal_element(V, scale=0.1)

dU = [V for i in [0,1,2,3,]]

JdU = dft.jacobian(U, U1, dU)

In [25]:
beta = g.default.get_float("--beta", 10.0)
seed = g.default.get("--seed", "hmc-pure-gauge")
ntherm = g.default.get_int("--ntherm", 10)
n = g.default.get_int("--n", 10)
nwrite = g.default.get_int("--nwrite", 10)
g.default.set_verbose("omf4")

# conjugate momenta
mom = g.group.cartesian(U)

# Log
g.message(f"Lattice = {grid.fdimensions}")
g.message("Actions:")
# action for conj. momenta
a0 = g.qcd.scalar.action.mass_term()
g.message(f" - {a0.__name__}")

# wilson action
a1 = g.qcd.gauge.action.wilson(beta)
g.message(f" - {a1.__name__}")


def hamiltonian():
    return a0(mom) + a1(U)


# molecular dynamics
sympl = g.algorithms.integrator.symplectic

ip = sympl.update_p(mom, lambda: a1.gradient(U, U))
iq = sympl.update_q(U, lambda: a0.gradient(mom, mom))

# integrator
mdint = sympl.OMF4(25, ip, iq)
g.message(f"Integration scheme:\n{mdint}")

# metropolis
metro = g.algorithms.markov.metropolis(rng)

# MD units
tau = 2.0
g.message(f"tau = {tau} MD units")


def hmc(tau, mom):
    rng.normal_element(mom)
    accrej = metro(U)
    h0 = hamiltonian()
    mdint(tau)
    h1 = hamiltonian()
    return [accrej(h1, h0), h1 - h0]


# thermalization
for i in range(1, 11):
    h = []
    timer = g.timer("hmc")
    for _ in range(ntherm // 10):
        timer("trajectory")
        h += [hmc(tau, mom)]
    h = np.array(h)
    timer()
    g.message(f"{i*10} % of thermalization completed")
    g.message(timer)
    g.message(
        f"Plaquette = {g.qcd.gauge.plaquette(U)}, Acceptance = {np.mean(h[:,0]):.2f}, |dH| = {np.mean(np.abs(h[:,1])):.4e}"
    )

# production
history = []
for i in range(n):
    history += [hmc(tau, mom)]
    P = g.qcd.gauge.plaquette(U)
    g.message(f"Trajectory {i}, P={P}")
    
history = np.array(history)
g.message(f"Acceptance rate = {np.mean(history[:,0]):.2f}")
g.message(f"<|dH|> = {np.mean(np.abs(history[:,1])):.4e}")

GPT :    5585.597271 s : Lattice = [4, 4, 4, 4]
GPT :    5585.597940 s : Actions:
GPT :    5585.598246 s :  - mass_term(m^-1 = 1.0)
GPT :    5585.598548 s :  - wilson(10.0)
GPT :    5585.599309 s : Integration scheme:
                       : OMF4(25, P, Q)
                       :   P(0.0033593261051506774, 0)
                       :   Q(0.01015914043364238, 0)
                       :   P(0.027289461342876364, 0)
                       :   Q(-0.0012921147061079868, 0)
                       :   P(-0.010648787448027042, 0)
                       :   Q(0.022265948544931215, 0)
                       :   P(-0.010648787448027042, 0)
                       :   Q(-0.0012921147061079868, 0)
                       :   P(0.027289461342876364, 0)
                       :   Q(0.01015914043364238, 0)
                       :   P(0.006718652210301355, 0)
                       :   Q(0.01015914043364238, 0)
                       :   P(0.027289461342876364, 0)
                       :   Q(-0.0012

In [ ]:
trace_U(U)

In [7]:
V = g.copy(U)
traces = []

tau = 1e-2
its = 200

def ft0(U):
    return ftg(U, eps=-tau)
    #return gtf(U)

fr = g.algorithms.optimize.fletcher_reeves
ls2 = g.algorithms.optimize.line_search_quadratic

dft = g.qcd.gauge.smear.differentiable_field_transformation(
    U,
    ft0,
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=its, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=its, restartlen=10),
    g.algorithms.optimize.non_linear_cg(
        maxiter=its, eps=1e-15, step=1e-1, line_search=ls2, beta=fr
    ),
)

steps = 5

plaqi = g.qcd.gauge.plaquette(V)

for i in range(steps):
    V = dft.inverse(V)
    traces.append(trace_U(V))

for i in range(steps):
    V = ftg(V, eps=-tau)
    traces.append(trace_U(V))

plaqf = g.qcd.gauge.plaquette(V)

print('diff plaq = ', plaqf-plaqi)

GPT :      60.522805 s : non_linear_cg: iteration 0: f(x) = 1.193742361023711e-02, |df|/sqrt(dof) = 1.330726e-02, beta = 0, step = 0.9232258848134469
GPT :      62.291466 s : non_linear_cg: iteration 10: f(x) = 3.103814781192328e-25, |df|/sqrt(dof) = 1.046690e-14, beta = 0.004190812520148747, step = 0.9350050657987338
GPT :      62.462103 s : non_linear_cg: max_abs_step adjustment for step = 1.0115645202910515
GPT :      62.484367 s : non_linear_cg: converged in 12 iterations: f(x) = 3.048343666296428e-25, |df|/sqrt(dof) = 5.624428e-16
GPT :      62.691686 s : non_linear_cg: iteration 0: f(x) = 1.095825749463009e-02, |df|/sqrt(dof) = 1.282828e-02, beta = 0, step = 0.9210297356563499
GPT :      64.467963 s : non_linear_cg: iteration 10: f(x) = 3.019095390177160e-25, |df|/sqrt(dof) = 1.039880e-14, beta = 0.004358814818304177, step = 0.9359156936567052
GPT :      64.660231 s : non_linear_cg: max_abs_step adjustment for step = 1.0054273221909424
GPT :      64.684062 s : non_linear_cg: conv

In [ ]:
plt.plot(traces[:-1])

In [15]:
W = g.copy(U)

fr = g.algorithms.optimize.fletcher_reeves
ls2 = g.algorithms.optimize.line_search_quadratic

def ft0(U):
    return ftg(U, eps=-1e-2)
    #return gtf(U)

dft = g.qcd.gauge.smear.differentiable_field_transformation(
    W,
    ft0,
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.optimize.non_linear_cg(
        maxiter=100, eps=1e-15, step=1e-1, line_search=ls2, beta=fr
    ),
)

dfm = dft.diffeomorphism()
ald = dft.action_log_det_jacobian()


In [16]:
mom = [g.group.cartesian(w) for w in W]
mom_prime = g.copy(mom)
rng.normal_element(mom_prime)

Wft = dfm(W)

mom = dfm.jacobian(W, Wft, mom_prime) # this appears to have fixed it???

mom2 = g.copy(mom)
g.message("Action log det jac:", ald(W + mom2))

GPT :     119.476237 s : fgcr: converged in 11 iterations;  computed squared residual 2.787557e-25 / 4.101855e-23;  true squared residual 2.789322e-25 / 4.101855e-23
GPT :     120.252519 s : fgcr: converged in 11 iterations;  computed squared residual 2.432427e-25 / 4.023726e-23;  true squared residual 2.435532e-25 / 4.023726e-23
GPT :     120.253139 s : Action log det jac: 4023.72620629567


In [17]:
V = g.mcolor(grid) # V is a SU(3) matrix on every lattice site
rng.normal_element(V, scale=0.1)

dU = [V for i in [0,1,2,3,]]

a = ald.gradient(W + mom2, W)

GPT :     121.684255 s : fgcr: converged in 11 iterations;  computed squared residual 2.787557e-25 / 4.101855e-23;  true squared residual 2.789322e-25 / 4.101855e-23
GPT :     122.460988 s : fgcr: converged in 11 iterations;  computed squared residual 2.432427e-25 / 4.023726e-23;  true squared residual 2.435532e-25 / 4.023726e-23


In [18]:
a[0][:][0]

array([[ 0.03554669+0.j        ,  0.09149298-0.07797728j,
        -0.02256771-0.01908549j],
       [ 0.09149298+0.07797728j,  0.01067945+0.j        ,
        -0.04541029+0.03135485j],
       [-0.02256771+0.01908549j, -0.04541029-0.03135485j,
        -0.04622614+0.j        ]])

In [19]:
a1 = g.qcd.gauge.action.wilson(beta=10.0)
a1.gradient(W, W)[0][:][0]

array([[ 0.39109043+0.j        , -2.04250842+0.53813427j,
         0.19311676-1.18475304j],
       [-2.04250842-0.53813427j, -1.36858503+0.j        ,
         0.65017378-0.1735966j ],
       [ 0.19311676+1.18475304j,  0.65017378+0.1735966j ,
         0.97749461+0.j        ]])

In [21]:
V = dft.inverse(W)

GPT :     127.526114 s : non_linear_cg: iteration 0: f(x) = 1.193742361023711e-02, |df|/sqrt(dof) = 1.330726e-02, beta = 0, step = 0.9232258848134469
GPT :     129.177069 s : non_linear_cg: iteration 10: f(x) = 3.103814781192328e-25, |df|/sqrt(dof) = 1.046690e-14, beta = 0.004190812520148747, step = 0.9350050657987338
GPT :     129.342316 s : non_linear_cg: max_abs_step adjustment for step = 1.0115645202910515
GPT :     129.363943 s : non_linear_cg: converged in 12 iterations: f(x) = 3.048343666296428e-25, |df|/sqrt(dof) = 5.624428e-16


In [22]:
V

[lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double)]